<a href="https://colab.research.google.com/github/anupbagale/python-for-ai-ml/blob/main/02-data-structures/01_lesson.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 2: Data Structures

In Module 1 we used lists and dictionaries without thinking about them much. This module looks at them properly, along with tuples, sets and the `collections` module.

There are two goals. The first is knowing what each structure can do, so you stop writing ten lines for something that takes one. The second, which matters more, is knowing how fast each one is. Choosing a list where you needed a set is one of the most common reasons Python data code is slow, and the difference isn't small. It can be the difference between a script that takes two seconds and one that takes an hour.

**Contents**

1. Lists in depth
2. Tuples
3. Dictionaries
4. Sets
5. How fast is each one?
6. The `collections` module
7. Top-k with `heapq`
8. Choosing the right structure
9. Putting it together: building a vocabulary for a text model

## 1. Lists in depth

A list is an ordered, mutable sequence. You already know the basics, so let's go through the parts people get wrong.

### Common methods

In [1]:
losses = [0.9, 0.7, 0.55]

losses.append(0.5)        # add one item to the end
print(losses)

losses.extend([0.45, 0.43])   # add every item from another list
print(losses)

losses.insert(0, 1.2)     # insert at a position
print(losses)

last = losses.pop()       # remove and return the last item
print(last, losses)

first = losses.pop(0)     # remove and return the item at a position
print(first, losses)

[0.9, 0.7, 0.55, 0.5]
[0.9, 0.7, 0.55, 0.5, 0.45, 0.43]
[1.2, 0.9, 0.7, 0.55, 0.5, 0.45, 0.43]
0.43 [1.2, 0.9, 0.7, 0.55, 0.5, 0.45]
1.2 [0.9, 0.7, 0.55, 0.5, 0.45]


A classic mix-up is `append` vs `extend`. `append` adds its argument as a single item, even if that argument is a list:

In [2]:
a = [1, 2]
a.append([3, 4])
print(a)       # a list inside a list

b = [1, 2]
b.extend([3, 4])
print(b)       # items added one by one

[1, 2, [3, 4]]
[1, 2, 3, 4]


A few more that come up often:

In [3]:
labels = ["cat", "dog", "cat", "bird", "cat"]

print(labels.count("cat"))   # how many times it appears
print(labels.index("dog"))   # position of the first match

labels.remove("cat")         # removes the FIRST "cat" only
print(labels)

3
1
['dog', 'cat', 'bird', 'cat']


`index` and `remove` raise a `ValueError` if the item isn't there, so check with `in` first if you're not sure.

### sort vs sorted

`list.sort()` sorts the list in place and returns `None`. `sorted()` returns a new sorted list and leaves the original alone.

In [4]:
scores = [0.82, 0.91, 0.77]

result = scores.sort()
print(result)    # None, a common source of bugs
print(scores)

scores = [0.82, 0.91, 0.77]
ranked = sorted(scores, reverse=True)
print(ranked, scores)

None
[0.77, 0.82, 0.91]
[0.91, 0.82, 0.77] [0.82, 0.91, 0.77]


If you ever write `x = my_list.sort()` and then get `None` errors later, this is why.

Both accept a `key` function that says what to sort by. To sort by several things, return a tuple from the key. Python compares tuples item by item, so it sorts by the first value and uses the second to break ties:

In [5]:
runs = [
    ("run_c", 0.91, 120),
    ("run_a", 0.91, 95),
    ("run_b", 0.88, 60),
]

# highest accuracy first, then fastest training time
best_first = sorted(runs, key=lambda r: (-r[1], r[2]))
for run in best_first:
    print(run)

('run_a', 0.91, 95)
('run_c', 0.91, 120)
('run_b', 0.88, 60)


The minus sign flips the order for accuracy only, so it sorts high to low while time still sorts low to high. This trick works for numbers. For strings you'd sort twice instead, relying on the fact that Python's sort is *stable*: items that compare equal keep their original order.

### Unpacking with *

You can unpack a list into variables, and a starred name collects whatever is left over:

In [6]:
header, *rows = ["id,label", "1,cat", "2,dog", "3,bird"]
print(header)
print(rows)

first, *middle, last = [10, 20, 30, 40, 50]
print(first, middle, last)

id,label
['1,cat', '2,dog', '3,bird']
10 [20, 30, 40] 50


### The list-of-lists trap

Multiplying a list repeats its items. That's fine for numbers, but watch what happens with a list of lists:

In [7]:
grid = [[0] * 3] * 3
print(grid)

grid[0][0] = 1
print(grid)

[[0, 0, 0], [0, 0, 0], [0, 0, 0]]
[[1, 0, 0], [1, 0, 0], [1, 0, 0]]


We changed one cell and a whole column changed. That's because `[[0] * 3] * 3` doesn't create three lists. It creates one list and puts three references to it in the outer list. It's the same aliasing issue from Module 1.

The fix is to create a new inner list each time, with a comprehension:

In [8]:
grid = [[0] * 3 for _ in range(3)]
grid[0][0] = 1
print(grid)

[[1, 0, 0], [0, 0, 0], [0, 0, 0]]


### Shallow vs deep copies

`.copy()` makes a *shallow* copy: a new outer list, but the items inside are the same objects. For a list of numbers that's all you need. For nested structures it isn't:

In [9]:
import copy

config = {"model": {"layers": [64, 32]}, "lr": 0.001}

shallow = config.copy()
shallow["lr"] = 0.01                       # top level: independent
shallow["model"]["layers"].append(16)      # nested: shared!

print(config)

{'model': {'layers': [64, 32, 16]}, 'lr': 0.001}


Changing `lr` in the copy didn't affect the original, but appending to the nested list did, because both dictionaries point to the same inner dict. When you need a fully independent copy of a nested structure, use `copy.deepcopy`:

In [10]:
config = {"model": {"layers": [64, 32]}, "lr": 0.001}

deep = copy.deepcopy(config)
deep["model"]["layers"].append(16)

print(config)
print(deep)

{'model': {'layers': [64, 32]}, 'lr': 0.001}
{'model': {'layers': [64, 32, 16]}, 'lr': 0.001}


This comes up when you run several experiments that start from one base config and tweak it. Without a deep copy, the tweaks from one experiment leak into the next.

## 2. Tuples

A tuple is like a list that can't be changed after it's created.

In [11]:
point = (3.0, 4.0)
print(point[0])

try:
    point[0] = 5.0
except TypeError as e:
    print("error:", e)

3.0
error: 'tuple' object does not support item assignment


The parentheses are mostly optional. It's the commas that make a tuple. That's why this works:

In [12]:
shape = 28, 28, 3
print(shape, type(shape))

height, width, channels = shape     # unpacking
print(height, width, channels)

a, b = 1, 2
a, b = b, a      # swap without a temporary variable
print(a, b)

(28, 28, 3) <class 'tuple'>
28 28 3
2 1


One gotcha: a tuple with one item needs a trailing comma. Without it, the parentheses are just grouping.

In [13]:
not_a_tuple = (5)
a_tuple = (5,)
print(type(not_a_tuple), type(a_tuple))

<class 'int'> <class 'tuple'>


You'll see `(5,)` all the time as a shape in NumPy and PyTorch, for a one-dimensional array of 5 values.

### Why use tuples at all?

Two reasons. First, they signal intent. A tuple says "this is a fixed group of values that belong together", like a shape, an RGB colour or a `(features, label)` pair, while a list says "this is a collection that might grow".

Second, and more practically, tuples can be used as dictionary keys and set members, and lists can't. Only immutable objects can be keys, because the dictionary relies on a key never changing. This lets you count things like pairs of words:

In [14]:
pair_counts = {}
pair_counts[("new", "york")] = 12
pair_counts[("machine", "learning")] = 30
print(pair_counts[("machine", "learning")])

try:
    pair_counts[["new", "york"]] = 1
except TypeError as e:
    print("error:", e)

30
error: unhashable type: 'list'


### namedtuple

When a tuple has more than two or three fields, `point[2]` stops being readable. A `namedtuple` lets you access fields by name while still being a tuple:

In [15]:
from collections import namedtuple

Prediction = namedtuple("Prediction", ["label", "score"])

p = Prediction("cat", 0.93)
print(p)
print(p.label, p.score)

label, score = p          # still unpacks like a normal tuple
print(label, score)

Prediction(label='cat', score=0.93)
cat 0.93
cat 0.93


For anything more complex, Module 3 covers dataclasses, which are usually a better fit.

## 3. Dictionaries

A dictionary maps keys to values. It's probably the most important data structure in Python: configs, label maps, vocabularies, JSON data, model state in PyTorch (`model.state_dict()` is a dictionary) and even Python's own variables are stored in dictionaries.

In [16]:
model = {"name": "resnet18", "params": 11_700_000, "pretrained": True}

print(model["name"])
model["epochs"] = 10         # add a key
model["params"] = 11_689_512 # update a key
del model["pretrained"]      # remove a key
print(model)

resnet18
{'name': 'resnet18', 'params': 11689512, 'epochs': 10}


### Missing keys

Looking up a key that doesn't exist raises a `KeyError`. There are two ways to deal with that: check with `in` first, or use `.get()`, which returns a default instead of raising.

In [17]:
print("lr" in model)            # in checks the keys
print(model.get("lr"))          # None if missing
print(model.get("lr", 0.001))   # or your own default

try:
    model["lr"]
except KeyError as e:
    print("KeyError:", e)

False
None
0.001
KeyError: 'lr'


`.pop(key, default)` removes a key and returns its value, which is handy for pulling an option out of a config before passing the rest on:

In [18]:
options = {"lr": 0.01, "momentum": 0.9, "verbose": True}
verbose = options.pop("verbose", False)
print(verbose, options)

True {'lr': 0.01, 'momentum': 0.9}


### Looping over a dictionary

In [19]:
class_counts = {"cat": 120, "dog": 95, "bird": 40}

for name in class_counts:               # keys
    print(name)

for count in class_counts.values():     # values
    print(count)

for name, count in class_counts.items():   # both
    print(f"{name}: {count}")

cat
dog
bird
120
95
40
cat: 120
dog: 95
bird: 40


Since Python 3.7, dictionaries keep keys in the order they were inserted. You can rely on that.

One rule: don't add or remove keys while looping over a dictionary. Python raises an error if the size changes during iteration. Loop over `list(d)` instead if you need to delete things as you go.

### Merging dictionaries

A really common pattern is having default settings and overriding some of them. There are two ways to merge:

In [20]:
defaults = {"lr": 0.001, "batch_size": 32, "epochs": 10}
overrides = {"lr": 0.01, "epochs": 50}

config = {**defaults, **overrides}   # works in any modern Python
print(config)

config = defaults | overrides        # Python 3.9+
print(config)

{'lr': 0.01, 'batch_size': 32, 'epochs': 50}
{'lr': 0.01, 'batch_size': 32, 'epochs': 50}


In both cases, when a key appears in both, the one on the right wins. Neither changes the original dictionaries. `defaults.update(overrides)` does the same merge but modifies `defaults` in place.

### Nested dictionaries

Configs are usually nested:

In [21]:
config = {
    "model": {"type": "mlp", "hidden": [128, 64], "dropout": 0.2},
    "optimizer": {"name": "adam", "lr": 3e-4},
    "data": {"batch_size": 64},
}

print(config["optimizer"]["lr"])
print(config.get("scheduler", {}).get("step_size", 10))

0.0003
10


The second line uses `.get` with an empty dict as the default, so the chain doesn't crash when `"scheduler"` is missing. You'll write a cleaner version of this in the exercises.

### Grouping with setdefault

`setdefault(key, default)` returns the value for a key, inserting the default first if the key is missing. It's useful for grouping:

In [22]:
samples = [("img1", "cat"), ("img2", "dog"), ("img3", "cat"), ("img4", "bird")]

by_label = {}
for filename, label in samples:
    by_label.setdefault(label, []).append(filename)

print(by_label)

{'cat': ['img1', 'img3'], 'dog': ['img2'], 'bird': ['img4']}


It works, but it's a bit cryptic. Section 6 shows `defaultdict`, which does the same thing more readably.

## 4. Sets

A set is an unordered collection of unique items. Think of it as a dictionary with only keys.

In [23]:
tags = {"nlp", "vision", "nlp", "audio"}
print(tags)              # the duplicate is gone, order isn't guaranteed

tags.add("rl")
tags.discard("audio")    # no error if missing (remove() would raise)
print(tags)
print("nlp" in tags)

{'vision', 'audio', 'nlp'}
{'rl', 'vision', 'nlp'}
True


Watch out: `{}` creates an empty **dictionary**, not an empty set. Use `set()`.

In [24]:
print(type({}), type(set()))

<class 'dict'> <class 'set'>


### Removing duplicates

`set(items)` is the quickest way to remove duplicates, but it throws away the order. If order matters, use `dict.fromkeys`, which keeps the first occurrence of each item:

In [25]:
words = ["the", "cat", "sat", "on", "the", "mat", "cat"]

print(set(words))                   # unordered
print(list(dict.fromkeys(words)))   # first-seen order kept

{'sat', 'cat', 'the', 'on', 'mat'}
['the', 'cat', 'sat', 'on', 'mat']


### Set operations

This is where sets get really useful. They support the operations from math class:

In [26]:
train_vocab = {"the", "cat", "sat", "on", "mat"}
test_vocab = {"the", "dog", "sat", "on", "log"}

print(train_vocab | test_vocab)   # union: in either
print(train_vocab & test_vocab)   # intersection: in both
print(test_vocab - train_vocab)   # difference: in test but not train
print(train_vocab ^ test_vocab)   # symmetric difference: in exactly one

{'sat', 'log', 'cat', 'the', 'dog', 'on', 'mat'}
{'sat', 'the', 'on'}
{'log', 'dog'}
{'cat', 'dog', 'log', 'mat'}


`test_vocab - train_vocab` is a real question in NLP: which words in the test data did the model never see during training? Those are your out-of-vocabulary words.

Another real use is checking for **data leakage**. If the same example appears in both your training and test sets, your test accuracy is inflated, because the model is being tested on something it memorised. With sets, the check is one line:

In [27]:
train_ids = {101, 102, 103, 104, 105, 106}
test_ids = {105, 106, 107, 108}

leaked = train_ids & test_ids
if leaked:
    print(f"warning: {len(leaked)} samples in both train and test: {sorted(leaked)}")

Duplicates across splits happen more than you'd think, especially with scraped data, where the same image or sentence shows up several times.

## 5. How fast is each one?

So far we've looked at what each structure does. Now let's look at how fast it does it, because this is where real performance problems come from.

### An experiment

Let's check whether a value is in a collection, using a list and a set of the same size. We'll search for a value that isn't there, which is the worst case for a list, since it has to look at every item before giving up.

In [28]:
import timeit

for n in [1_000, 10_000, 100_000, 1_000_000]:
    as_list = list(range(n))
    as_set = set(as_list)

    t_list = timeit.timeit(lambda: -1 in as_list, number=100) / 100
    t_set = timeit.timeit(lambda: -1 in as_set, number=100) / 100

    print(f"n={n:>9,}   list: {t_list * 1e6:9.1f} µs   set: {t_set * 1e6:5.2f} µs")

n=    1,000   list:       4.9 µs   set:  0.06 µs
n=   10,000   list:      50.0 µs   set:  0.05 µs
n=  100,000   list:     487.3 µs   set:  0.07 µs


n=1,000,000   list:    4533.6 µs   set:  0.05 µs


Your exact numbers will be different, but the pattern will be the same. Every time the list gets 10 times bigger, the search takes about 10 times longer. The set takes about the same time no matter how big it is.

### Why?

A list doesn't know where anything is. To answer `x in my_list`, it compares `x` with the first item, then the second, and so on. Twice as many items means twice the work. We say that's **O(n)**: the time grows in proportion to the size `n`.

Sets and dictionaries are built on **hash tables**. When you add an item, Python computes a number from it (its *hash*) and uses that number to decide where to store it. To look something up, it computes the hash again and goes straight to that spot. The size of the collection barely matters. That's **O(1)**, or constant time.

This is also why set items and dictionary keys must be immutable: if a key could change after being stored, its hash would change and Python would look for it in the wrong place.

### Why this matters in practice

One `in` check on a list is fast enough either way. The problem is putting it inside a loop:

In [29]:
import random

random.seed(0)
vocab_list = [f"word{i}" for i in range(20_000)]
vocab_set = set(vocab_list)
tokens = [f"word{random.randint(0, 40_000)}" for _ in range(20_000)]

t_list = timeit.timeit(lambda: [t for t in tokens if t in vocab_list], number=1)
t_set = timeit.timeit(lambda: [t for t in tokens if t in vocab_set], number=1)

print(f"list: {t_list:.3f} s")
print(f"set:  {t_set:.4f} s")
print(f"set is about {t_list / t_set:,.0f}x faster")

list: 1.877 s
set:  0.0022 s
set is about 872x faster


Same result, same code, except one variable is a set instead of a list. With a real vocabulary of 50,000 words and a corpus of millions of tokens, the list version would take hours.

**Rule of thumb:** if you check `x in something` inside a loop, `something` should be a set or a dict.

### Adding to the front

Lists are fast at the end and slow at the front. `append` and `pop()` are O(1), but `insert(0, x)` and `pop(0)` have to shift every other item along by one, which is O(n). For a queue where you add at one end and remove from the other, use `collections.deque` (more on it in the next section):

In [30]:
from collections import deque

def fill_list(n):
    items = []
    for i in range(n):
        items.insert(0, i)

def fill_deque(n):
    items = deque()
    for i in range(n):
        items.appendleft(i)

n = 50_000
print(f"list.insert(0, x):   {timeit.timeit(lambda: fill_list(n), number=1):.3f} s")
print(f"deque.appendleft(x): {timeit.timeit(lambda: fill_deque(n), number=1):.4f} s")

list.insert(0, x):   0.230 s
deque.appendleft(x): 0.0014 s


### Cost summary

| Operation | list | dict / set | deque |
|---|---|---|---|
| `x in c` | O(n), slow | O(1), fast | O(n), slow |
| get by position `c[i]` | O(1) | n/a | O(n) in the middle |
| get by key `d[k]` | n/a | O(1) | n/a |
| add at the end | O(1) | O(1) (`add`, `d[k] = v`) | O(1) |
| add or remove at the front | O(n), slow | n/a | O(1) |
| sort | O(n log n) | n/a | n/a |

You don't need to memorise Big-O notation to use this. The practical version is: lists are for ordered data you go through or index by position, dicts and sets are for looking things up, and deques are for queues.

## 6. The collections module

The standard library's `collections` module has a few specialised containers that make common jobs much shorter.

### Counter

`Counter` counts things. You wrote this yourself in Module 1 as `count_labels`, and here it is built in:

In [31]:
from collections import Counter

labels = ["cat", "dog", "cat", "bird", "cat", "dog"]
counts = Counter(labels)

print(counts)
print(counts["cat"])
print(counts["fish"])          # missing keys give 0, not a KeyError
print(counts.most_common(2))   # the n most common, sorted
print(counts.total())          # sum of all counts (3.10+)

Counter({'cat': 3, 'dog': 2, 'bird': 1})
3
0
[('cat', 3), ('dog', 2)]
6


Counters can be updated and combined, which is handy when you're counting across batches or files:

In [32]:
batch1 = Counter(["cat", "dog", "cat"])
batch2 = Counter(["dog", "dog", "bird"])

print(batch1 + batch2)

running = Counter()
for batch in [["a", "b"], ["b", "c"], ["b"]]:
    running.update(batch)
print(running)

Counter({'dog': 3, 'cat': 2, 'bird': 1})
Counter({'b': 3, 'a': 1, 'c': 1})


A quick class balance check is a single line, and it's one of the first things you should look at in any classification dataset:

In [33]:
dataset_labels = ["spam"] * 150 + ["ham"] * 850
for label, n in Counter(dataset_labels).most_common():
    print(f"{label:5s} {n:4d}  ({n / len(dataset_labels):.0%})")

ham    850  (85%)
spam   150  (15%)


### defaultdict

A `defaultdict` is a dictionary that creates a default value the first time you access a missing key. You tell it what kind of default to make by passing a function, usually a type like `list` or `int`:

In [34]:
from collections import defaultdict

samples = [("img1", "cat"), ("img2", "dog"), ("img3", "cat"), ("img4", "bird")]

by_label = defaultdict(list)
for filename, label in samples:
    by_label[label].append(filename)   # no need to check if the key exists

print(dict(by_label))

{'cat': ['img1', 'img3'], 'dog': ['img2'], 'bird': ['img4']}


Compare that to the `setdefault` version in section 3. Same result, much easier to read.

You can nest them. Here's a word count per class, which is exactly what the spam filter in the mini project needs:

In [35]:
messages = [
    ("win a free prize now", "spam"),
    ("are we still meeting now", "ham"),
    ("free prize claim now", "spam"),
]

word_counts = defaultdict(Counter)
for text, label in messages:
    word_counts[label].update(text.split())

print(word_counts["spam"].most_common(3))
print(word_counts["ham"].most_common(3))

[('free', 2), ('prize', 2), ('now', 2)]
[('are', 1), ('we', 1), ('still', 1)]


One thing to be careful about: just *reading* a missing key from a `defaultdict` creates it. `if by_label["fish"]:` adds an empty `"fish"` list. Use `"fish" in by_label` to check without creating anything.

### deque

A `deque` ("deck") is a double-ended queue: fast to add and remove at both ends. Give it a `maxlen` and it becomes a fixed-size window that automatically drops the oldest item when a new one comes in. That's perfect for a moving average of recent losses:

In [36]:
recent = deque(maxlen=3)

for step, loss in enumerate([0.9, 0.8, 0.85, 0.6, 0.55, 0.5]):
    recent.append(loss)
    avg = sum(recent) / len(recent)
    print(f"step {step}: window={list(recent)}  avg={avg:.3f}")

step 0: window=[0.9]  avg=0.900
step 1: window=[0.9, 0.8]  avg=0.850
step 2: window=[0.9, 0.8, 0.85]  avg=0.850
step 3: window=[0.8, 0.85, 0.6]  avg=0.750
step 4: window=[0.85, 0.6, 0.55]  avg=0.667
step 5: window=[0.6, 0.55, 0.5]  avg=0.550


In reinforcement learning, the same idea is used for an *experience replay buffer*: the agent stores its last N experiences in a deque and trains on random samples from it. Old experiences fall off the end automatically.

In [37]:
random.seed(1)
buffer = deque(maxlen=1000)

for step in range(5000):
    experience = (f"state{step}", "action", 1.0, f"state{step + 1}")
    buffer.append(experience)

print(len(buffer))        # never more than 1000
print(buffer[0][0])       # the oldest one left
batch = random.sample(list(buffer), 4)
print([e[0] for e in batch])

1000
state4000
['state4137', 'state4582', 'state4867', 'state4821']


## 7. Top-k with heapq

Classifiers often output a score for every class, and you want the top few. You could sort everything and take the first `k`, but if you only want a handful out of thousands, `heapq.nlargest` is the tool for the job, and it reads more clearly too:

In [38]:
import heapq

class_scores = {"cat": 0.61, "dog": 0.22, "fox": 0.09, "wolf": 0.05, "bird": 0.03}

top3 = heapq.nlargest(3, class_scores.items(), key=lambda kv: kv[1])
for name, score in top3:
    print(f"{name:5s} {score:.2f}")

cat   0.61
dog   0.22
fox   0.09


This is "top-k accuracy" territory. ImageNet results are often reported as top-5 accuracy: the prediction counts as correct if the true class is anywhere in the model's five highest scores.

`heapq` also gives you a *priority queue*: a list where you can always pop the smallest item quickly. `heappush` adds an item, `heappop` removes the smallest. Both are O(log n), much faster than re-sorting a list after every change:

In [39]:
queue = []
heapq.heappush(queue, (0.31, "sample_7"))
heapq.heappush(queue, (0.05, "sample_2"))
heapq.heappush(queue, (0.18, "sample_9"))

while queue:
    confidence, sample = heapq.heappop(queue)
    print(f"label next: {sample} (confidence {confidence})")

label next: sample_2 (confidence 0.05)
label next: sample_9 (confidence 0.18)
label next: sample_7 (confidence 0.31)


That example is the idea behind *active learning*: send the samples the model is least confident about to a human for labelling first.

## 8. Choosing the right structure

| You need to... | Use |
|---|---|
| keep items in order, go through them, index by position | `list` |
| store a fixed record, or use a group of values as a key | `tuple` / `namedtuple` |
| look things up by a key | `dict` |
| check membership fast, remove duplicates, compare collections | `set` |
| count things | `Counter` |
| group items by a key | `defaultdict(list)` |
| a queue, or a sliding window of the last N items | `deque` |
| the k largest or smallest items, or a priority queue | `heapq` |

When in doubt, ask yourself what operation you'll do most often, and pick the structure that makes that operation fast.

## 9. Putting it together: building a vocabulary for a text model

Neural networks can't read words. Before text goes into a model, every word gets replaced by an integer id, and those ids are looked up in an embedding table. The mapping from words to ids is called the **vocabulary**, and building one uses almost everything from this module.

Here's what we need:

1. Split sentences into tokens (words).
2. Count how often each word appears, and drop very rare words. Rare words take up space in the vocabulary and the model can't learn much from them anyway.
3. Reserve a few special tokens: `<pad>` for padding short sentences, and `<unk>` for any word not in the vocabulary.
4. Build word-to-id and id-to-word mappings.
5. Encode sentences as lists of ids, and decode them back.
6. Pad a batch of sentences to the same length, since models process batches as rectangular blocks.

In [40]:
corpus = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "the cat chased the dog",
    "a bird sat on the cat",
]

def tokenize(text):
    return text.lower().split()

counts = Counter(token for sentence in corpus for token in tokenize(sentence))
print(counts.most_common())

[('the', 7), ('cat', 3), ('sat', 3), ('on', 3), ('dog', 2), ('mat', 1), ('log', 1), ('chased', 1), ('a', 1), ('bird', 1)]


Now build the mappings. Words are added in order of frequency, which is a common convention (the most frequent words get the smallest ids). Ties are broken alphabetically so the result is the same every run.

In [41]:
PAD, UNK = "<pad>", "<unk>"

def build_vocab(counts, min_freq=1):
    words = [w for w, c in counts.items() if c >= min_freq]
    words.sort(key=lambda w: (-counts[w], w))
    id_to_word = [PAD, UNK] + words
    word_to_id = {w: i for i, w in enumerate(id_to_word)}
    return word_to_id, id_to_word

word_to_id, id_to_word = build_vocab(counts, min_freq=2)
print(word_to_id)

{'<pad>': 0, '<unk>': 1, 'the': 2, 'cat': 3, 'on': 4, 'sat': 5, 'dog': 6}


With `min_freq=2`, words that appear only once ("mat", "log", "chased", "a", "bird") are left out. Notice the two data structures: a **dict** for word to id, because we look up words, and a **list** for id to word, because ids are positions. Each direction uses the structure that makes its lookup O(1).

Encoding uses `.get` with the `<unk>` id as the default, so unknown words don't crash anything:

In [42]:
def encode(text):
    unk_id = word_to_id[UNK]
    return [word_to_id.get(token, unk_id) for token in tokenize(text)]

def decode(ids):
    return " ".join(id_to_word[i] for i in ids)

ids = encode("the cat sat on the hamster")
print(ids)
print(decode(ids))

[2, 3, 5, 4, 2, 1]
the cat sat on the <unk>


"hamster" was never seen, so it became `<unk>`. Finally, padding a batch so every sequence has the same length:

In [43]:
def pad_batch(sequences):
    pad_id = word_to_id[PAD]
    longest = max(len(s) for s in sequences)
    return [s + [pad_id] * (longest - len(s)) for s in sequences]

batch = [encode(s) for s in ["the cat sat", "the dog sat on the mat", "a cat"]]
for row in pad_batch(batch):
    print(row)

[2, 3, 5, 0, 0, 0]
[2, 6, 5, 4, 2, 1]
[1, 3, 0, 0, 0, 0]


That's a working text preprocessing pipeline. Real tokenizers (like the ones in Hugging Face) split words into smaller pieces and handle punctuation, but the core structure is exactly this: a counter, a dict, a list and a couple of special tokens.

## Summary

- `append` adds one item, `extend` adds many. `sort()` returns `None`, `sorted()` returns a new list.
- Sort by several things with a tuple key: `key=lambda r: (-r.score, r.time)`.
- `[[0] * 3] * 3` repeats one inner list. Use a comprehension to make separate lists.
- `.copy()` is shallow. Use `copy.deepcopy` for nested configs.
- Tuples are immutable, so they can be dict keys and set members. Lists can't.
- Use `.get(key, default)` for optional keys, and `{**a, **b}` or `a | b` to merge dicts.
- Sets remove duplicates and support `|`, `&`, `-` and `^`. Use them to find out-of-vocabulary words and train/test leakage.
- `x in list` is O(n). `x in set` and `x in dict` are O(1). If you check membership in a loop, use a set.
- `Counter` counts, `defaultdict` groups, `deque(maxlen=n)` keeps the last n items, `heapq.nlargest` gets the top k.

Next, open `02_exercises.ipynb`.